# PROJECT TITLE

**Question:** what this notebook answers.

**Rule of this project:** the CRS of every layer is checked before any area, distance or buffer calculation, and every result states the CRS used and how many features had missing or invalid values.

## 0. Setup

Works both locally and in Google Colab. In Colab the repository is cloned so that `src/` can be imported.

In [ ]:
import sys, os, subprocess

REPO = "geo-project-template"  # replace with the repository name
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Colab only opens the notebook: clone the repo once and work from its root.
    # Safe to run again or after "Restart session".
    if os.path.basename(os.getcwd()) != REPO:
        if not os.path.exists(REPO):
            subprocess.run(["git", "clone", f"https://github.com/albertolozada/{REPO}.git"], check=True)
        os.chdir(REPO)
    # Libraries not preinstalled in Colab
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "contextily", "matplotlib-scalebar"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

sys.path.insert(0, os.getcwd())

import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
from src.geo_checks import check_crs, to_metric, area_ha
from src.maps import plot_aoi_map

## 1. Load data

Replace this example with the real layers of the project (downloaded into `data/`). The example is a small polygon in geographic coordinates (EPSG:4326, degrees).

In [ ]:
aoi = gpd.GeoDataFrame(
    {"name": ["example_area"]},
    geometry=[box(-60.6, -21.6, -60.4, -21.4)],
    crs="EPSG:4326",
)
aoi

## 2. Check the CRS of every layer

In [ ]:
check_crs(aoi, "aoi")

The layer is in degrees, so areas computed directly would be wrong. Reproject to a metric CRS first and **write down which one was used**.

In [ ]:
aoi_m = to_metric(aoi)
print("CRS used for measurements:", aoi_m.crs.to_string())
check_crs(aoi_m, "aoi (metric)")

## 3. Analysis

In [ ]:
aoi_m["area_ha"] = area_ha(aoi_m).round(1)
aoi_m[["name", "area_ha"]]

## 4. Map

### Location map

Study area over a satellite basemap, with scale bar, north arrow and basemap attribution. Use it as `outputs/maps/preview.png` (the image under the title of the README) or in the *Study area* section. Needs internet access to download the basemap tiles.

In [ ]:
plot_aoi_map(
    aoi,
    out_png="outputs/maps/location_map.png",
    basemap="satellite",   # "satellite" | "light" | "osm"
    title="Study area",
    dpi=120,
)
plt.show()

In [ ]:
os.makedirs("outputs/maps", exist_ok=True)
ax = aoi_m.plot(figsize=(6, 6), edgecolor="black", facecolor="#2f6b4f", alpha=0.6)
ax.set_title(f"Example area ({aoi_m.crs.to_string()})")
plt.savefig("outputs/maps/example_map.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Results and limitations

*Fill in this section by hand after running the analysis: the code computes the numbers, this section explains what they mean. Use the CRS and the counts printed by `check_crs` above.*

- **Main result:**
- **CRS used:**
- **Completeness:** features with missing or invalid values
- **Limitations:**